In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import xgboost as xgb
import joblib

# Load dataset
file_path = "cleaned_amazon_data (2).csv"  # Update with correct path
data = pd.read_csv(file_path)

# Drop unnecessary columns
data.drop(columns=['product_name', 'product_link'], inplace=True, errors='ignore')

# Convert rating and rating_count to numeric
data['rating'] = pd.to_numeric(data['rating'], errors='coerce')
data['rating_count'] = data['rating_count'].str.extract('(\d+)').astype(float)

# Fill missing values
data['discounted_price'].fillna(data['actual_price'] * 0.8, inplace=True)
data['actual_price'].fillna(data['discounted_price'] / 0.8, inplace=True)
numeric_data = data.select_dtypes(include=np.number)
data[numeric_data.columns] = numeric_data.fillna(numeric_data.median())

# Encode categorical variables (One-Hot Encoding)
data = pd.get_dummies(data, columns=['Category'], drop_first=True)

# Define Features and Target
X = data.drop(columns=['discounted_price'])  # Features
y = data['discounted_price']  # Target Variable

# Split data into training and test sets (80%-20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and train the XGBoost model
model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, learning_rate=0.1, max_depth=6)
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Calculate performance metrics
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse}")
print(f"R² Score: {r2}")

# Save the trained model
joblib.dump(model, "xgboost_model.pkl")
print("Model saved successfully as 'xgboost_model.pkl'")

<ipython-input-1-5283f50b300c>:20: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['discounted_price'].fillna(data['actual_price'] * 0.8, inplace=True)
<ipython-input-1-5283f50b300c>:21: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value,

RMSE: 1234.0665691465056
R² Score: 0.9554229014734954
Model saved successfully as 'xgboost_model.pkl'
